# Text Generation
This notebook walks through the **full lifecycle of text generation** using a causal language model (CLM):
tokenization → logit prediction → greedy decoding → autoregressive loop

## Learning Objectives
By the end of this lab you should be able to:
- Load a causal LM and tokenizer from Hugging Face
- Explain the shape and meaning of model logits
- Generate the next token using greedy decoding
- multi-step generation without KV caching

---
## 1. Setup
| Library | Purpose |
|---|---|
| `torch` | Tensor operations and the deep learning runtime |
| `transformers` | Hugging Face API to load pretrained models and tokenizers |


Running on GPU dramatically reduces per-token latency because matrix multiplications are parallelised across thousands of CUDA cores.


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer  # HuggingFace model hub

# Fix random seeds for reproducibility across runs
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Use GPU if available; fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch version:", torch.__version__)
print("Using device   :", device)


PyTorch version: 2.10.0+cu128
Using device   : cuda


---
## 2. Load a Language Model

**What is a Causal Language Model (CLM)?**  
A CLM is trained to predict the next token given all previous tokens.  
It is called *causal* because it can only look *left* (past tokens), never right (future tokens).  
This is enforced by a **causal attention mask** — a lower-triangular matrix that blocks future positions.

**DistilGPT-2 architecture at a glance:**

```
Token IDs  →  Token Embedding (50 257 × 768)
           +  Positional Embedding (1 024 × 768)
           →  6 × Transformer Blocks  [Attention + FFN + LayerNorm]
           →  Final LayerNorm
           →  LM Head  Linear(768 → 50 257)   ← produces logits
```
**`model.eval()`** switches off dropout randomness used only during training,  
ensuring deterministic outputs during inference.


In [ ]:
model_name = "distilgpt2"            # lighter version of GPT-2, same concepts apply
print("Loading model:", model_name)

# AutoTokenizer: automatically picks the right tokenizer class for the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-2 has no pad token by default; set it to EOS so batching works correctly
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# AutoModelForCausalLM: loads weights for next-token prediction (LM head included)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move all model parameters to the selected device (GPU or CPU)
model.to(device)

# Disable dropout and other training-only behaviours for deterministic inference
model.eval()

print("Model class :", model.__class__.__name__)
print("Vocab size  :", tokenizer.vocab_size)


Loading model: distilgpt2


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model class : GPT2LMHeadModel
Vocab size  : 50257


### Model Architecture

Key layers to notice:
- **`wte` (word token embedding)** — maps each token ID to a 768-dim vector
- **`wpe` (word position embedding)** — adds positional information (up to 1 024 positions)
- **`h.0 … h.5`** — 6 identical transformer blocks, each with self-attention + feed-forward
- **`lm_head`** — final linear layer projecting 768 dims → 50 257 vocab scores (logits)


In [ ]:
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


## 3. Tokenize a Prompt

**What does a tokenizer do?**  
Neural networks work with numbers, not text.  
The tokenizer converts a raw string into a sequence of **integer token IDs** drawn from a fixed vocabulary.  
DistilGPT-2 uses **Byte-Pair Encoding (BPE)** — words are split into subword pieces to keep the vocabulary manageable (~50 k entries) while handling any input text.

```
"The quick brown fox jumped over the"
  ↓  BPE tokenisation
[464, 2068, 7586, 21831, 11687, 625, 262]
```

The tokenizer also returns an **attention mask** — a tensor of 1s and 0s indicating  
which positions the model should attend to (1 = real token, 0 = padding).  
For a single prompt with no padding, all values are 1.

**Tensor shape convention:** `(batch_size, sequence_length)`  
Here: batch = 1 (one prompt), sequence = 7 tokens → shape `(1, 7)`


In [ ]:
prompt = "The quick brown fox jumped over the"  # input text to the model

# tokenizer(...) returns a dict with 'input_ids' and 'attention_mask'
# return_tensors="pt" means return PyTorch tensors instead of plain lists
inputs = tokenizer(prompt, return_tensors="pt")

# Move input tensors to the same device as the model
inputs = {k: v.to(device) for k, v in inputs.items()}

print("Raw tokenizer output:", inputs)
print()
print("Prompt          :", prompt)
print("Input IDs       :", inputs["input_ids"])        # integer token IDs
print("Attention mask  :", inputs["attention_mask"])   # 1 = attend, 0 = ignore
print("Input shape     :", tuple(inputs["input_ids"].shape))  # (batch, seq_len)


Raw tokenizer output: {'input_ids': tensor([[  464,  2068,  7586, 21831, 11687,   625,   262]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

Prompt          : The quick brown fox jumped over the
Input IDs       : tensor([[  464,  2068,  7586, 21831, 11687,   625,   262]], device='cuda:0')
Attention mask  : tensor([[1, 1, 1, 1, 1, 1, 1]], device='cuda:0')
Input shape     : (1, 7)


###  Observe
- The batch dimension (first) is always 1 here — we're processing a single prompt.
- The sequence length (second) tells you how many tokens the prompt was split into.
- Each integer maps to exactly one entry in the tokenizer's vocabulary.

In [ ]:
# Convert token IDs back to their subword string pieces for inspection
token_pieces = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())
for pos, (piece, tid) in enumerate(zip(token_pieces, inputs["input_ids"][0].tolist())):
    print(f"  Position {pos}  →  token piece: {piece!r:15s}  id: {tid}")


  Position 0  →  token piece: 'The'            id: 464
  Position 1  →  token piece: 'Ġquick'         id: 2068
  Position 2  →  token piece: 'Ġbrown'         id: 7586
  Position 3  →  token piece: 'Ġfox'           id: 21831
  Position 4  →  token piece: 'Ġjumped'        id: 11687
  Position 5  →  token piece: 'Ġover'          id: 625
  Position 6  →  token piece: 'Ġthe'           id: 262


---
## 4. Predict the Next Token from Logits

**What are logits?**  
After the model processes the input tokens, the `lm_head` layer outputs a raw score  
for every vocabulary token at every sequence position.  
These raw scores are called **logits** (unnormalised log-probabilities).

**Logits tensor shape:** `(batch_size, sequence_length, vocab_size)`  
→ Here: `(1, 7, 50 257)` — for each of the 7 input positions, 50 257 scores are produced.

**Why not return probabilities directly?**  
Logits give the caller full control:
- `argmax(logits)` → **greedy decoding** (always pick the highest-score token)
- `softmax(logits)` then `multinomial(probs)` → **sampling** (stochastic, creative)

**Why only the final position?**  
Positions 0–5 predict tokens already in the prompt (known → discarded at inference).  
Only the **last position** predicts a genuinely new token.

```
logits[0, 0, :]  →  predicts token after "The"      ← already known
logits[0, 1, :]  →  predicts token after "quick"    ← already known
...
logits[0, 6, :]  →  predicts token after "the"      ← NEW — use this
```

**Greedy decoding:** `next_token = argmax(logits[0, -1, :])`


In [ ]:
# Disable gradient tracking — we are doing inference, not training
with torch.no_grad():
    outputs = model(**inputs)   # forward pass

# outputs.logits shape: (batch_size=1, seq_len=7, vocab_size=50257)
logits = outputs.logits
print("Logits shape:", tuple(logits.shape))   # (1, 7, 50257)

# Focus on the LAST position: it predicts the next new token
# logits[batch=0, position=-1, all_vocab]
last_logits = logits[0, -1, :]

# Greedy decoding: pick the vocabulary index with the highest raw score
next_token_id = torch.argmax(last_logits)

# Decode the integer ID back to a human-readable string
next_token = tokenizer.decode(next_token_id)

print("Next token id   :", int(next_token_id))
print("Greedy next token:", repr(next_token))


Logits shape: (1, 7, 50257)
Next token id   : 13990
Greedy next token: ' fence'


### Softmax and Top-K Inspection

Raw logits are hard to interpret directly.  
Converting them to probabilities via **softmax** makes relative confidence readable:

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

We can also inspect the **top-k** most likely next tokens to understand the model's uncertainty.


In [ ]:
# Retrieve the top-10 most likely next tokens and their logit scores
top_k = torch.topk(last_logits, k=10)

# Decode each top-k token ID and pair it with its score
top_tokens = [
    (tokenizer.decode(idx), round(float(score), 3))
    for score, idx in zip(top_k.values, top_k.indices)
]

print(f"{'Rank':<6} {'Token':<15} {'Logit Score'}")
for rank, (token, score) in enumerate(top_tokens, 1):
    print(f"  {rank:<4} {token!r:<15} {score}")


Rank   Token           Logit Score
  1    ' fence'        -52.88
  2    ' top'          -53.511
  3    ' wall'         -53.845
  4    ' head'         -54.315
  5    ' tree'         -54.436
  6    ' edge'         -54.541
  7    ' railing'      -54.959
  8    ' ground'       -55.055
  9    ' back'         -55.188
  10   ' door'         -55.31


1.  **Why do we use `logits[0, -1, :]` and not `logits[0, 0, :]`?**
    We use `logits[0, -1, :]` because the model is a Causal Language Model (CLM), which predicts the *next* token based on all *previous* tokens. The logits for the last position (`-1`) correspond to the prediction for the token that comes *after* the entire input sequence. `logits[0, 0, :]` would predict the token after the first token in the input, which is not what we want for autoregressive generation.

2. **How does the gap between the top-1 and top-2 logit scores reflect the  
    model's confidence?**
    A larger gap between the top-1 and top-2 logit scores indicates higher confidence from the model in its top prediction. A small gap suggests that the model sees multiple tokens as almost equally likely, reflecting lower confidence or ambiguity in its prediction for the next token.

## 5. Append the Predicted Token (One Autoregressive Step)

**Autoregressive generation** means feeding the newly predicted token back into the  
model as part of the input for the next step. The sequence grows by one token each time:

```
Step 0 input:  [The, quick, brown, fox, jumped, over, the]          → predicts: fence
Step 1 input:  [The, quick, brown, fox, jumped, over, the, fence]   → predicts: .
Step 2 input:  [The, quick, brown, fox, jumped, over, the, fence, .] → ...
```

To do this we **concatenate** the new token ID to `input_ids`,  
and extend the `attention_mask` by one extra `1` so the model attends to the new token.

This is the simplest (but not the most efficient) way to do it —  
**KV caching** avoids re-processing old tokens


In [ ]:
# Concatenate the new token ID to the existing input_ids along the sequence dimension (dim=1)
# next_token_id.view(1,1) reshapes scalar → (batch=1, seq=1) to match input_ids shape
new_input_ids = torch.cat([inputs["input_ids"], next_token_id.view(1, 1)],dim=1)   # concatenate along sequence length axis


# Extend the attention mask with a '1' for the newly added token
new_attention_mask = torch.cat([inputs["attention_mask"],torch.ones((1, 1), dtype=inputs["attention_mask"].dtype, device=device)],dim=1)

next_inputs = {
    "input_ids": new_input_ids,
    "attention_mask": new_attention_mask,
}

print("Old input_ids shape:", tuple(inputs["input_ids"].shape))          # (1, 7)
print("New input_ids shape:", tuple(next_inputs["input_ids"].shape))     # (1, 8)
print()
print("Decoded sequence after appending predicted token:")
print(" ", tokenizer.decode(next_inputs["input_ids"][0]))


Old input_ids shape: (1, 7)
New input_ids shape: (1, 8)

Decoded sequence after appending predicted token:
  The quick brown fox jumped over the fence


###  Observe
The sequence length increased from 7 → 8.  
Each future call to `model(**next_inputs)` will process all 8 tokens again from scratch.  
This full recomputation is optimised with KV caching.


---
## 6. Autoregressive Generation Loop (Without KV Cache)

We now wrap the single-step logic into a loop that generates `max_new_tokens` tokens one by one.

**At each step:**
1. Run a full forward pass over the **entire** current sequence
2. Extract logits at the final position
3. Apply `argmax` (greedy) to pick the next token
4. Append the token and repeat

**Why does latency grow with each step?**  
Self-attention computes a score between every pair of tokens in the sequence.  
If the sequence has *n* tokens, attention requires **O(n²)** operations.  
Each new token makes the sequence one longer, so every subsequent step is more expensive than the last.

```
Step 1: attend over  7 tokens  →  7² = 49 pairs
Step 2: attend over  8 tokens  →  8² = 64 pairs
Step 3: attend over  9 tokens  →  9² = 81 pairs
...
Step 10: attend over 17 tokens → 17² = 289 pairs
```

In [ ]:
def generate_token(model, tokenizer, model_inputs):
    """Single greedy decode step — returns the ID of the most likely next token."""
    with torch.no_grad():                          # no gradients needed at inference
        outputs = model(**model_inputs)            # full forward pass over all tokens

    logits = outputs.logits                        # shape: (1, seq_len, vocab_size)
    # argmax over vocab dimension at the last sequence position
    next_token_id = torch.argmax(logits[:, -1, :], dim=-1)
    return next_token_id                           # shape: (batch_size,)


def greedy_generate(model, tokenizer, prompt, max_new_tokens=10):
    """
    Generate `max_new_tokens` tokens autoregressively WITHOUT KV caching.
    Re-processes the full sequence at every step.
    Returns: list of generated token strings, per-step latencies, full decoded text.
    """
    # Tokenise the prompt and move tensors to device
    model_inputs = tokenizer(prompt, return_tensors="pt")
    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}

    generated_tokens = []   # store decoded token strings
    durations_s = []        # store wall-clock time per step

    for _ in range(max_new_tokens):
        t0 = time.time()                                    # start timer
        next_token_id = generate_token(model, tokenizer, model_inputs)
        durations_s.append(time.time() - t0)               # record elapsed time

        # Decode the integer token ID to a string and store it
        generated_tokens.append(tokenizer.decode(next_token_id[0]))

        # Grow the sequence: append new token to input_ids and extend attention mask
        model_inputs = {
            "input_ids": torch.cat([model_inputs["input_ids"], next_token_id.view(1, 1)], dim=1),
            "attention_mask": torch.cat([model_inputs["attention_mask"],
                 torch.ones((1, 1), dtype=model_inputs["attention_mask"].dtype, device=device)],
                dim=1),
        }

    # Decode the full sequence (prompt + all generated tokens) to readable text
    full_text = tokenizer.decode(model_inputs["input_ids"][0])
    return generated_tokens, durations_s, full_text


In [ ]:
# Run greedy generation WITHOUT KV caching and measure step-by-step latency
generated_tokens, durations_s, full_text = greedy_generate(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=10,
)

print("Generated tokens :", generated_tokens)
print("Full text        :\n ", full_text)
print("Total time (s)   :", round(sum(durations_s), 4))


Generated tokens : [' fence', ' and', ' ran', ' over', ' the', ' fence', '.', '\n', '\n', '\n']
Full text        :
  The quick brown fox jumped over the fence and ran over the fence.



Total time (s)   : 0.1272


Why might generation become slower as the sequence grows?
Two compounding reasons — one computational, one architectural.

Reason 1 — Attention is O(n²)
The self-attention mechanism computes a score between every pair of tokens. If the sequence has n tokens, attention requires O(n²) operations. Double the sequence length → quadruple the computation.

Reason 2 — Autoregressive generation is inherently sequential
Each new token depends on all tokens before it, so you cannot generate token 5 until tokens 1–4 are ready. Every step requires a full forward pass, and each forward pass is slower than the last because the sequence is longer